# GMD-normalised photon-energy resonance search

This notebook investigates the photon-energy dependence of the gas-phase glycine electron spectrum measured during the FLASH beamtime. The main aim is to search for the carbon 1s to inner-valence resonance and compare its location and spectral behaviour with the earlier glycine measurements of Schwickert *et al.* (Science Advances, 2022), where the carbon 1s to 10a' transition was reported near 272.7 eV.

The analysis uses shot-resolved eTOF hits, converts them to electron kinetic energy, and normalises each selected shot group to its summed FEL pulse energy measured by the GMD. This is an ordinary electron-yield analysis, not an electron-electron coincidence or covariance measurement.

## Imports and plotting configuration

The project configuration selects the FLASH data paths. `FLASH_ENV=remote` is retained from the working analysis for use on the ASAP3 server.

In [ ]:
import os
import sys
import re
import json
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

%matplotlib inline

os.environ.setdefault("FLASH_ENV", "remote")


def find_repo_root(start=None):
    '''Find the repository containing analysis/scripts.'''
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "analysis" / "scripts").exists():
            return path
    raise RuntimeError("Could not find repository root containing analysis/scripts")


repo_root = find_repo_root()
scripts_dir = repo_root / "analysis" / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import config
from compute_aggregates import load_aggregates

AggregatesData = load_aggregates.__globals__["AggregatesData"]

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.grid": False,
})

## Analysis configuration

The two GMD intervals stay inside the approximately linear low-pulse-energy regime identified in the shot-resolved control tests. Floating-point and integer filename families are kept separate because they belong to different datasets with substantially different signal levels. The 276.5 eV row is excluded to match the final presentation selection.

In [ ]:
AGG_DIR = Path(config.COMBINED_DIR)
AGG_GLOB = "glycine_WL_scan_*eV_aggregates_etof1.h5"

MIN_ENERGY = 270.0
MAX_ENERGY = 288.0
EXCLUDED_ENERGIES = (276.5,)

LOW_GMD_RANGE = (0.6, 1.3)
HIGH_GMD_RANGE = (1.3, 2.0)

TOF_RANGE = (800, 1100)
KE_XLIM = (180, 280)
CONTROL_ENERGY_EV = 270.0

CHUNK_TRAINS = 1000
MIN_SHOTS_PER_GROUP = 500
MIN_SUM_GMD_PER_GROUP = 100.0

FRAC_DENOM_MIN = 1e-4
USE_ONLY_USABLE_ROWS = True

CANDIDATE_FAMILY = "floating"
CANDIDATE_KE_WINDOWS = [
    (190, 200),
    (200, 210),
    (210, 220),
    (190, 210),
    (200, 220),
    (195, 215),
    (190, 220),
]
BACKGROUND_COMPARISON_KE_RANGE = (180, 190)
BASELINE_ENERGY_RANGES = [(270.0, 272.5), (278.0, 283.0)]
REFERENCE_ENERGY_EV = 274.0
PEAK_KE_RANGE = (190, 220)
PEAK_ENERGY_WINDOW = (273.0, 275.0)

CALIB_PATH = (
    Path(config.COMBINED_DIR).parent
    / "Calibrations"
    / "etof_energy_calibration_t0_scan.json"
)

## Data loading and calibration

The lightweight aggregate loader is used only to discover each scan run, read its TOF grid and trimming metadata, and locate the corresponding normal shot-resolved HDF5 file. The spectra below are reconstructed from the shot-by-shot `gmd` and `tofs_e` arrays; aggregate electron spectra are not used in the final resonance comparison.

The original TOF bins map to unequal kinetic-energy widths. Spectra and absolute difference maps are therefore divided by each calibrated KE-bin width and plotted as yield densities in counts / uJ / eV. Dimensionless ratios are not divided by KE-bin width.

In [ ]:
def load_aggregates_etof(path):
    '''Load eTOF aggregate metadata without the large covariance arrays.'''
    path = Path(path)
    with h5py.File(path, "r") as handle:
        metadata = {}
        for key in handle.attrs:
            value = handle.attrs[key]
            metadata[key] = value.tolist() if isinstance(value, np.ndarray) else value

        def optional(key):
            return handle[key][:] if key in handle else None

        return AggregatesData(
            G=handle["G"][:],
            GtG=optional("GtG"),
            n_per_bin=handle["n_per_bin"][:],
            gmd_edges=handle["gmd_edges"][:],
            D=optional("D"),
            tof_edges=optional("tof_edges"),
            DtD=None,
            DtG=None,
            A=None,
            AtA=None,
            AtD=None,
            AtG=None,
            vls_pixels=optional("vls_pixels"),
            background=optional("background"),
            C=None,
            CtC=None,
            DtC=None,
            CtG=None,
            ion_tof_edges=optional("ion_tof_edges"),
            z_edges=optional("z_edges"),
            nominal_energies=optional("nominal_energies"),
            metadata=metadata,
        )


ENERGY_RE = re.compile(
    r"^glycine_WL_scan_(\d+(?:\.\d+)?)eV_aggregates_etof1$"
)


def energy_from_name(path):
    match = ENERGY_RE.search(Path(path).stem)
    if match is None:
        raise ValueError(f"Could not parse photon energy from {Path(path).name}")
    return float(match.group(1))


def scan_family_from_filename(path):
    match = ENERGY_RE.search(Path(path).stem)
    if match is None:
        return "unknown"
    return "floating" if "." in match.group(1) else "integer"


files = []
for path in sorted(AGG_DIR.glob(AGG_GLOB)):
    energy = energy_from_name(path)
    if not (MIN_ENERGY <= energy <= MAX_ENERGY):
        continue
    if any(np.isclose(energy, excluded) for excluded in EXCLUDED_ENERGIES):
        continue
    files.append((path, energy))

files.sort(key=lambda item: (item[1], item[0].name))
if not files:
    raise FileNotFoundError(f"No files matched {AGG_DIR / AGG_GLOB}")

with open(CALIB_PATH, "r", encoding="utf-8") as handle:
    calib = json.load(handle)

best_t0 = calib["t0_tof_units"]
best_slope = calib["slope_eV_tof_unit2"]
best_E0 = calib["E0_eV"]


def tof_to_ke(tof):
    tof = np.asarray(tof, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ke = best_slope / (tof - best_t0) ** 2 + best_E0
    return np.where(tof > best_t0, ke, np.nan)


def half_step_edges(values):
    values = np.asarray(values, dtype=float)
    if values.size == 1:
        return np.array([values[0] - 0.5, values[0] + 0.5])
    midpoints = 0.5 * (values[:-1] + values[1:])
    return np.concatenate([
        [2 * values[0] - midpoints[0]],
        midpoints,
        [2 * values[-1] - midpoints[-1]],
    ])


def tof_spectrum_to_ke_density(tof_edges, spectrum, tof_range):
    '''Convert counts/uJ/TOF-bin to counts/uJ/eV on the native grid.'''
    tof_edges = np.asarray(tof_edges, dtype=float)
    spectrum = np.asarray(spectrum, dtype=float)
    tof_centres = 0.5 * (tof_edges[:-1] + tof_edges[1:])

    mask = np.ones_like(tof_centres, dtype=bool)
    if tof_range[0] is not None:
        mask &= tof_centres >= tof_range[0]
    if tof_range[1] is not None:
        mask &= tof_centres <= tof_range[1]

    indices = np.where(mask)[0]
    if indices.size == 0:
        raise ValueError(f"No TOF bins inside range {tof_range}")

    lo = int(indices[0])
    hi = int(indices[-1]) + 1
    ke_edges = tof_to_ke(tof_edges[lo : hi + 1])
    ke_centres = tof_to_ke(tof_centres[lo:hi])
    density = spectrum[lo:hi].astype(float) / np.abs(np.diff(ke_edges))

    good = np.isfinite(ke_centres) & np.isfinite(density)
    order = np.argsort(ke_centres[good])
    return ke_centres[good][order], density[good][order]


def tof_map_to_ke_density(tof_edges, data, tof_range):
    '''Convert the final axis of a TOF map to native unequal-bin KE density.'''
    tof_edges = np.asarray(tof_edges, dtype=float)
    tof_centres = 0.5 * (tof_edges[:-1] + tof_edges[1:])

    mask = np.ones_like(tof_centres, dtype=bool)
    if tof_range[0] is not None:
        mask &= tof_centres >= tof_range[0]
    if tof_range[1] is not None:
        mask &= tof_centres <= tof_range[1]

    indices = np.where(mask)[0]
    if indices.size == 0:
        raise ValueError(f"No TOF bins inside range {tof_range}")

    lo = int(indices[0])
    hi = int(indices[-1]) + 1
    ke_edges = tof_to_ke(tof_edges[lo : hi + 1])
    data_ke = np.asarray(data, dtype=float)[..., lo:hi]
    data_ke = data_ke / np.abs(np.diff(ke_edges))

    if ke_edges[0] > ke_edges[-1]:
        ke_edges = ke_edges[::-1]
        data_ke = data_ke[..., ::-1]

    return ke_edges, data_ke

In [ ]:
def keep_mask_like_compute_aggregates_from_h5(handle, trim_start=0, trim_end=0):
    '''Reproduce the train-level selection used during aggregate generation.'''
    if "between_tdc_files" not in handle:
        return np.ones(handle["gmd"].shape[0], dtype=bool)

    between_tdc_files = handle["between_tdc_files"][:].astype(bool)
    good_positions = np.where(~between_tdc_files)[0]
    if trim_end > 0:
        good_positions = good_positions[trim_start:-trim_end]
    else:
        good_positions = good_positions[trim_start:]

    keep = np.zeros(between_tdc_files.shape[0], dtype=bool)
    keep[good_positions] = True
    return keep


def source_h5_from_aggregate_file(agg_path):
    agg = load_aggregates_etof(agg_path)
    source_h5 = Path(agg.metadata.get("input_h5", ""))
    if not source_h5.exists():
        raise FileNotFoundError(
            f"Source H5 not found for {Path(agg_path).name}: {source_h5}"
        )
    return source_h5, agg


def shot_group_spectrum_from_h5(
    source_h5,
    gmd_range,
    tof_edges,
    *,
    trim_start=0,
    trim_end=0,
    chunk_trains=1000,
):
    '''Return sum(counts)/sum(GMD) for a shot-resolved GMD interval.'''
    gmd_lo, gmd_hi = gmd_range
    total_counts = np.zeros(len(tof_edges) - 1, dtype=np.float64)
    total_gmd = 0.0
    total_shots = 0
    total_hits = 0

    with h5py.File(source_h5, "r") as handle:
        keep = keep_mask_like_compute_aggregates_from_h5(
            handle,
            trim_start=trim_start,
            trim_end=trim_end,
        )
        n_trains = handle["gmd"].shape[0]

        for start in range(0, n_trains, chunk_trains):
            stop = min(start + chunk_trains, n_trains)
            row_keep = keep[start:stop]
            if not row_keep.any():
                continue

            gmd = handle["gmd"][start:stop][row_keep]
            tofs = handle["tofs_e"][start:stop][row_keep]
            gmd_flat = gmd.ravel()
            tofs_flat = tofs.reshape(-1, tofs.shape[-1])

            selected = (
                np.isfinite(gmd_flat)
                & (gmd_flat >= gmd_lo)
                & (gmd_flat < gmd_hi)
            )
            if not selected.any():
                continue

            hits = tofs_flat[selected].ravel()
            hits = hits[hits > 0]
            histogram, _ = np.histogram(hits, bins=tof_edges)

            total_counts += histogram.astype(np.float64)
            total_gmd += float(np.nansum(gmd_flat[selected]))
            total_shots += int(np.count_nonzero(selected))
            total_hits += int(hits.size)

    with np.errstate(invalid="ignore", divide="ignore"):
        spectrum = total_counts / total_gmd
    spectrum[~np.isfinite(spectrum)] = np.nan

    return {
        "spectrum": spectrum,
        "counts": total_counts,
        "sum_gmd": total_gmd,
        "shots": total_shots,
        "hits": total_hits,
        "gmd_range": gmd_range,
    }


def low_high_from_aggregate_source(agg_path):
    source_h5, agg = source_h5_from_aggregate_file(agg_path)
    trim_start = int(agg.metadata.get("trim_start", 0))
    trim_end = int(agg.metadata.get("trim_end", 0))
    tof_edges = np.asarray(agg.tof_edges, dtype=float)

    low = shot_group_spectrum_from_h5(
        source_h5,
        LOW_GMD_RANGE,
        tof_edges,
        trim_start=trim_start,
        trim_end=trim_end,
        chunk_trains=CHUNK_TRAINS,
    )
    high = shot_group_spectrum_from_h5(
        source_h5,
        HIGH_GMD_RANGE,
        tof_edges,
        trim_start=trim_start,
        trim_end=trim_end,
        chunk_trains=CHUNK_TRAINS,
    )

    difference = high["spectrum"] - low["spectrum"]
    with np.errstate(invalid="ignore", divide="ignore"):
        fractional = 2.0 * difference / (high["spectrum"] + low["spectrum"])
    fractional[~np.isfinite(fractional)] = np.nan

    return {
        "source_h5": source_h5,
        "tof_edges": tof_edges,
        "low": low,
        "high": high,
        "diff": difference,
        "frac": fractional,
    }

## Load the wavelength scans

Each aggregate filename identifies one acquisition and its dataset family. Overlapping nominal photon energies from the floating-point and integer families are not merged: they were acquired under different conditions and are analysed with separate colour scales. Within each acquisition, the low- and high-GMD spectra are reconstructed from the normal shot-resolved HDF5 file.

A row is retained for plotting only when both GMD groups meet the minimum shot and summed-GMD requirements.

In [ ]:
full_rows = []
tof_edges_ref = None

for agg_path, energy in files:
    try:
        result = low_high_from_aggregate_source(agg_path)
    except (FileNotFoundError, KeyError, OSError) as exc:
        print(f"Skipping {agg_path.name}: {exc}")
        continue

    if tof_edges_ref is None:
        tof_edges_ref = result["tof_edges"]
    elif not np.array_equal(tof_edges_ref, result["tof_edges"]):
        print(f"Skipping {agg_path.name}: TOF edge mismatch")
        continue

    low = result["low"]
    high = result["high"]
    usable = (
        low["shots"] >= MIN_SHOTS_PER_GROUP
        and high["shots"] >= MIN_SHOTS_PER_GROUP
        and low["sum_gmd"] >= MIN_SUM_GMD_PER_GROUP
        and high["sum_gmd"] >= MIN_SUM_GMD_PER_GROUP
    )

    full_rows.append({
        "energy": float(energy),
        "path": agg_path,
        "source_h5": result["source_h5"],
        "family": scan_family_from_filename(agg_path),
        "low": low,
        "high": high,
        "diff": result["diff"],
        "frac": result["frac"],
        "usable": usable,
    })

if not full_rows:
    raise RuntimeError("No wavelength-scan rows could be reconstructed")

print("Loaded shot-resolved wavelength-scan rows:")
for family in ("floating", "integer"):
    family_rows = [row for row in full_rows if row["family"] == family]
    usable_rows = [row for row in family_rows if row["usable"]]
    if family_rows:
        energies = np.array([row["energy"] for row in usable_rows])
        print(
            f"  {family:8s}: {len(usable_rows):2d}/{len(family_rows):2d} usable, "
            f"{energies.min():.1f}-{energies.max():.1f} eV"
        )

## Low-energy GMD-normalisation control

At 270 eV the spectrum is expected to be dominated by ordinary one-photon ionisation. The upper panel compares

\[
S(E)=\frac{\sum_i N_i(E)}{\sum_i \mathrm{GMD}_i}
\]

for the two low-GMD intervals. The middle panel shows high minus low, and the lower panel shows the dimensionless high/low ratio. Agreement is not exact at every kinetic energy, but this low-GMD comparison avoids the strongly sublinear response observed at larger pulse energies.

These curves have no statistical error bars; their purpose is a direct spectral-shape control. Shot and summed-GMD totals are printed below the plot.

In [ ]:
control_candidates = [
    row for row in full_rows
    if row["family"] == "floating"
    and np.isclose(row["energy"], CONTROL_ENERGY_EV)
]
if len(control_candidates) != 1:
    raise ValueError(
        f"Expected one floating control row at {CONTROL_ENERGY_EV} eV, "
        f"found {len(control_candidates)}"
    )

control = control_candidates[0]
ke_low, y_low = tof_spectrum_to_ke_density(
    tof_edges_ref, control["low"]["spectrum"], TOF_RANGE
)
ke_high, y_high = tof_spectrum_to_ke_density(
    tof_edges_ref, control["high"]["spectrum"], TOF_RANGE
)
ke_diff, y_diff = tof_spectrum_to_ke_density(
    tof_edges_ref, control["diff"], TOF_RANGE
)

tof_centres = 0.5 * (tof_edges_ref[:-1] + tof_edges_ref[1:])
tof_mask = (tof_centres >= TOF_RANGE[0]) & (tof_centres <= TOF_RANGE[1])
ke_ratio = tof_to_ke(tof_centres[tof_mask])
with np.errstate(invalid="ignore", divide="ignore"):
    ratio = (
        control["high"]["spectrum"][tof_mask]
        / control["low"]["spectrum"][tof_mask]
    )
ratio_denom = (
    control["high"]["spectrum"][tof_mask]
    + control["low"]["spectrum"][tof_mask]
)
good = (
    np.isfinite(ke_ratio)
    & np.isfinite(ratio)
    & (ratio_denom >= FRAC_DENOM_MIN)
)
order = np.argsort(ke_ratio[good])
ke_ratio = ke_ratio[good][order]
ratio = ratio[good][order]

fig, axes = plt.subplots(
    3,
    1,
    figsize=(10, 9),
    sharex=True,
    constrained_layout=True,
    gridspec_kw={"height_ratios": [2.0, 1.0, 1.0]},
)

axes[0].plot(ke_low, y_low, lw=1.0, label=f"low {LOW_GMD_RANGE[0]}-{LOW_GMD_RANGE[1]} uJ")
axes[0].plot(ke_high, y_high, lw=1.0, label=f"high {HIGH_GMD_RANGE[0]}-{HIGH_GMD_RANGE[1]} uJ")
axes[0].set_ylabel("S(E), counts / uJ / eV")
axes[0].set_title(f"Shot-resolved low-GMD control at {control['energy']:.1f} eV")
axes[0].legend()

axes[1].plot(ke_diff, y_diff, lw=1.0, color="k")
axes[1].axhline(0.0, color="tab:red", lw=0.8, alpha=0.7)
axes[1].set_ylabel("high - low")

axes[2].plot(ke_ratio, ratio, lw=1.0, color="tab:purple")
axes[2].axhline(1.0, color="tab:red", lw=0.8, alpha=0.7)
axes[2].set_xlabel("electron kinetic energy (eV)")
axes[2].set_ylabel("high / low")
axes[2].set_xlim(KE_XLIM)
axes[2].set_ylim(0.5, 1.5)

for axis in axes:
    axis.grid(alpha=0.25)

plt.show()

for label in ("low", "high"):
    group = control[label]
    hits_per_gmd = group["hits"] / group["sum_gmd"]
    print(
        f"{label:4s}: shots={group['shots']:8d}, "
        f"sum GMD={group['sum_gmd']:10.5g} uJ, "
        f"hits/sum GMD={hits_per_gmd:.5g}"
    )

## Photon-energy / kinetic-energy difference maps

For every incident-energy run, the absolute map is

\[
\Delta S(h\nu,E)=S_{\mathrm{high}}(h\nu,E)-S_{\mathrm{low}}(h\nu,E).
\]

A contribution that scales linearly with FEL pulse energy should largely cancel after GMD normalisation. Positive structure can indicate a stronger high-GMD contribution, while negative structure indicates a smaller GMD-normalised high-GMD yield. The fractional map is shown only as a visual diagnostic and is masked where its denominator is small.

Photoelectron bands should move in kinetic energy as the photon energy changes; an Auger contribution would remain approximately fixed in kinetic energy. No residual-gas spectrum is subtracted in this final GMD analysis, matching the working presentation branch.

In [ ]:
def plot_family_difference_maps(
    full_rows,
    family,
    *,
    use_only_usable=True,
    diff_vlim=None,
    frac_vlim=0.5,
):
    rows = [
        row for row in full_rows
        if row["family"] == family and (row["usable"] or not use_only_usable)
    ]
    rows.sort(key=lambda row: (row["energy"], Path(row["path"]).name))
    if not rows:
        raise ValueError(f"No rows to plot for family={family!r}")

    energies = np.array([row["energy"] for row in rows], dtype=float)
    difference_tof = np.vstack([row["diff"] for row in rows])
    fractional_tof = np.vstack([row["frac"] for row in rows])
    denominator_tof = np.vstack([
        row["high"]["spectrum"] + row["low"]["spectrum"]
        for row in rows
    ])

    ke_edges, difference_ke = tof_map_to_ke_density(
        tof_edges_ref, difference_tof, TOF_RANGE
    )

    # Undo the density Jacobian for dimensionless quantities.
    ke_widths = np.abs(np.diff(ke_edges))
    _, fractional_density = tof_map_to_ke_density(
        tof_edges_ref, fractional_tof, TOF_RANGE
    )
    _, denominator_density = tof_map_to_ke_density(
        tof_edges_ref, denominator_tof, TOF_RANGE
    )
    fractional_ke = fractional_density * ke_widths[None, :]
    denominator_ke = denominator_density * ke_widths[None, :]

    fractional_ke[denominator_ke < FRAC_DENOM_MIN] = np.nan
    fractional_ke[np.abs(fractional_ke) > 2.0] = np.nan

    if diff_vlim is None:
        finite = difference_ke[np.isfinite(difference_ke)]
        diff_vlim = float(np.nanpercentile(np.abs(finite), 99.0))
    frac_vlim = min(float(frac_vlim), 2.0)

    energy_edges = half_step_edges(energies)
    diff_norm = TwoSlopeNorm(
        vmin=-diff_vlim, vcenter=0.0, vmax=diff_vlim
    )
    frac_norm = TwoSlopeNorm(
        vmin=-frac_vlim, vcenter=0.0, vmax=frac_vlim
    )

    fig, axes = plt.subplots(
        1, 2, figsize=(14, 5), sharey=True, constrained_layout=True
    )
    image_abs = axes[0].pcolormesh(
        ke_edges,
        energy_edges,
        difference_ke,
        shading="auto",
        cmap="RdBu_r",
        norm=diff_norm,
    )
    fig.colorbar(
        image_abs, ax=axes[0], label="high - low, counts / uJ / eV"
    )
    axes[0].set_title(
        f"{family}: Delta S\n"
        f"{HIGH_GMD_RANGE[0]}-{HIGH_GMD_RANGE[1]} minus "
        f"{LOW_GMD_RANGE[0]}-{LOW_GMD_RANGE[1]} uJ"
    )
    axes[0].set_xlabel("electron kinetic energy (eV)")
    axes[0].set_ylabel("incident photon energy (eV)")

    image_frac = axes[1].pcolormesh(
        ke_edges,
        energy_edges,
        fractional_ke,
        shading="auto",
        cmap="RdBu_r",
        norm=frac_norm,
    )
    fig.colorbar(
        image_frac,
        ax=axes[1],
        label="2(high - low) / (high + low)",
    )
    axes[1].set_title(f"{family}: fractional Delta S")
    axes[1].set_xlabel("electron kinetic energy (eV)")

    for axis in axes:
        axis.set_xlim(KE_XLIM)
    axes[0].set_ylim(np.nanmax(energy_edges), np.nanmin(energy_edges))
    plt.show()

    return fig, axes


# Independent scaling is intentional because the integer family has lower signal.
plot_family_difference_maps(
    full_rows,
    "floating",
    use_only_usable=USE_ONLY_USABLE_ROWS,
    diff_vlim=None,
    frac_vlim=0.6,
)

plot_family_difference_maps(
    full_rows,
    "integer",
    use_only_usable=USE_ONLY_USABLE_ROWS,
    diff_vlim=None,
    frac_vlim=0.5,
)

## Integrated candidate-window traces

Each point below is a KE lineout of one incident-energy row integrated over a fixed kinetic-energy window. Fractional overlap weights retain partially covered edge bins on the native unequal KE grid.

The error bars are propagated Poisson counting uncertainties from the raw low- and high-GMD counts. For each group, the weighted count variance is divided by the square of the group's summed GMD; low and high uncertainties are then combined in quadrature. These are not standard errors across shots.

For the lower panel, each KE-window trace has one inverse-variance-weighted scalar off-resonance mean subtracted. Its uncertainty is propagated into every baseline-subtracted point. The empirical off-resonance RMS is retained separately as a diagnostic of additional run-to-run scatter.

In [ ]:
def ke_window_weights_for_tof_bins(tof_edges, ke_range):
    '''Fractional overlap of each native calibrated bin with a KE window.'''
    ke_edges = tof_to_ke(tof_edges)
    lo = np.minimum(ke_edges[:-1], ke_edges[1:])
    hi = np.maximum(ke_edges[:-1], ke_edges[1:])
    width = hi - lo
    overlap = np.maximum(
        0.0,
        np.minimum(hi, ke_range[1]) - np.maximum(lo, ke_range[0]),
    )

    weights = np.zeros_like(width, dtype=float)
    good = np.isfinite(lo) & np.isfinite(hi) & (width > 0)
    weights[good] = overlap[good] / width[good]
    return weights


def integrate_group_with_uncertainty(group, tof_edges, ke_range):
    '''Integrate counts/sum(GMD) and propagate Poisson count variance.'''
    weights = ke_window_weights_for_tof_bins(tof_edges, ke_range)
    counts = np.asarray(group["counts"], dtype=float)
    sum_gmd = float(group["sum_gmd"])

    weighted_counts = float(np.nansum(weights * counts))
    variance_counts = float(
        np.nansum((weights ** 2) * np.maximum(counts, 0.0))
    )
    if sum_gmd <= 0:
        return np.nan, np.nan, weighted_counts

    value = weighted_counts / sum_gmd
    sigma = np.sqrt(variance_counts) / sum_gmd
    return value, sigma, weighted_counts


def build_window_traces(full_rows, family, ke_windows, *, use_only_usable=True):
    rows = [
        row for row in full_rows
        if row["family"] == family and (row["usable"] or not use_only_usable)
    ]
    rows.sort(key=lambda row: (row["energy"], Path(row["path"]).name))

    output = {
        "rows": rows,
        "energies": np.array([row["energy"] for row in rows], dtype=float),
        "delta": {},
        "sigma": {},
        "low": {},
        "high": {},
    }

    for ke_range in ke_windows:
        low_values = []
        high_values = []
        differences = []
        uncertainties = []

        for row in rows:
            low, sigma_low, _ = integrate_group_with_uncertainty(
                row["low"], tof_edges_ref, ke_range
            )
            high, sigma_high, _ = integrate_group_with_uncertainty(
                row["high"], tof_edges_ref, ke_range
            )
            low_values.append(low)
            high_values.append(high)
            differences.append(high - low)
            uncertainties.append(np.sqrt(sigma_low ** 2 + sigma_high ** 2))

        output["low"][ke_range] = np.array(low_values, dtype=float)
        output["high"][ke_range] = np.array(high_values, dtype=float)
        output["delta"][ke_range] = np.array(differences, dtype=float)
        output["sigma"][ke_range] = np.array(uncertainties, dtype=float)

    return output


def mask_energy_ranges(energies, ranges):
    mask = np.zeros_like(energies, dtype=bool)
    for lo, hi in ranges:
        mask |= (energies >= lo) & (energies <= hi)
    return mask


def weighted_baseline_mean(values, uncertainties, baseline_mask):
    values = np.asarray(values, dtype=float)
    uncertainties = np.asarray(uncertainties, dtype=float)
    good = (
        baseline_mask
        & np.isfinite(values)
        & np.isfinite(uncertainties)
        & (uncertainties > 0)
    )
    if np.count_nonzero(good) < 2:
        return np.nan, np.nan, np.nan, 0

    weights = 1.0 / uncertainties[good] ** 2
    mean = np.sum(weights * values[good]) / np.sum(weights)
    mean_error = np.sqrt(1.0 / np.sum(weights))
    rms = np.sqrt(np.nanmean((values[good] - mean) ** 2))
    return mean, mean_error, rms, int(np.count_nonzero(good))


def nearest_energy_index(energies, target):
    energies = np.asarray(energies, dtype=float)
    return int(np.nanargmin(np.abs(energies - target)))


candidate_traces = build_window_traces(
    full_rows,
    CANDIDATE_FAMILY,
    CANDIDATE_KE_WINDOWS + [BACKGROUND_COMPARISON_KE_RANGE],
    use_only_usable=USE_ONLY_USABLE_ROWS,
)
incident_energies = candidate_traces["energies"]
baseline_mask = mask_energy_ranges(
    incident_energies, BASELINE_ENERGY_RANGES
)

In [ ]:
fig, axes = plt.subplots(
    2, 1, figsize=(10, 8), sharex=True, constrained_layout=True
)

highlighted_windows = [(190, 220), (190, 210), (195, 215)]
colours = plt.get_cmap("viridis")(
    np.linspace(0, 1, len(CANDIDATE_KE_WINDOWS))
)
baseline_results = {}

for colour, ke_range in zip(colours, CANDIDATE_KE_WINDOWS):
    values = candidate_traces["delta"][ke_range]
    uncertainties = candidate_traces["sigma"][ke_range]
    off_mean, off_mean_error, off_rms, n_off = weighted_baseline_mean(
        values, uncertainties, baseline_mask
    )

    zeroed = values - off_mean
    zeroed_error = np.sqrt(uncertainties ** 2 + off_mean_error ** 2)
    baseline_results[ke_range] = {
        "off_mean": off_mean,
        "off_mean_err": off_mean_error,
        "off_rms": off_rms,
        "n_off": n_off,
        "zeroed": zeroed,
        "zeroed_err": zeroed_error,
    }

    highlighted = ke_range in highlighted_windows
    alpha = 1.0 if highlighted else 0.35
    linewidth = 1.7 if highlighted else 0.9
    zorder = 5 if highlighted else 2

    axes[0].errorbar(
        incident_energies,
        values,
        yerr=uncertainties,
        fmt="o-",
        ms=4,
        lw=linewidth,
        capsize=2,
        color=colour,
        alpha=alpha,
        zorder=zorder,
        label=f"{ke_range[0]}-{ke_range[1]} eV",
    )
    axes[1].errorbar(
        incident_energies,
        zeroed,
        yerr=zeroed_error,
        fmt="o-",
        ms=4,
        lw=linewidth,
        capsize=2,
        color=colour,
        alpha=alpha,
        zorder=zorder,
        label=f"{ke_range[0]}-{ke_range[1]} eV",
    )

comparison = candidate_traces["delta"][BACKGROUND_COMPARISON_KE_RANGE]
comparison_sigma = candidate_traces["sigma"][BACKGROUND_COMPARISON_KE_RANGE]
comparison_mean, comparison_mean_error, comparison_rms, comparison_n = (
    weighted_baseline_mean(comparison, comparison_sigma, baseline_mask)
)
comparison_zeroed = comparison - comparison_mean
comparison_zeroed_error = np.sqrt(
    comparison_sigma ** 2 + comparison_mean_error ** 2
)

axes[0].errorbar(
    incident_energies,
    comparison,
    yerr=comparison_sigma,
    fmt="s--",
    ms=4,
    lw=1.0,
    capsize=2,
    color="0.35",
    alpha=0.75,
    label=(
        f"comparison {BACKGROUND_COMPARISON_KE_RANGE[0]}-"
        f"{BACKGROUND_COMPARISON_KE_RANGE[1]} eV"
    ),
)
axes[1].errorbar(
    incident_energies,
    comparison_zeroed,
    yerr=comparison_zeroed_error,
    fmt="s--",
    ms=4,
    lw=1.0,
    capsize=2,
    color="0.35",
    alpha=0.75,
    label=(
        f"comparison {BACKGROUND_COMPARISON_KE_RANGE[0]}-"
        f"{BACKGROUND_COMPARISON_KE_RANGE[1]} eV"
    ),
)

for axis in axes:
    axis.axhline(0.0, color="k", lw=0.8, alpha=0.6)
    for lo, hi in BASELINE_ENERGY_RANGES:
        axis.axvspan(lo, hi, color="0.8", alpha=0.25)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8, ncols=2)

axes[0].set_ylabel("integrated Delta S, counts / uJ")
axes[0].set_title(f"{CANDIDATE_FAMILY}: raw candidate KE-window traces")
axes[1].set_xlabel("incident photon energy (eV)")
axes[1].set_ylabel("Delta S minus weighted off-resonance mean")
axes[1].set_title("Photon-energy-baseline-subtracted traces")

plt.show()

In [ ]:
reference_index = nearest_energy_index(
    incident_energies, REFERENCE_ENERGY_EV
)
reference_energy = incident_energies[reference_index]

print(
    "window        zeroed@ref      sigma_counting   off_rms         "
    "z_counting      z_with_rms"
)
summary_rows = []

for ke_range in CANDIDATE_KE_WINDOWS:
    values = candidate_traces["delta"][ke_range]
    uncertainties = candidate_traces["sigma"][ke_range]
    baseline = baseline_results[ke_range]

    zeroed_reference = values[reference_index] - baseline["off_mean"]
    sigma_counting = np.sqrt(
        uncertainties[reference_index] ** 2
        + baseline["off_mean_err"] ** 2
    )
    sigma_with_rms = np.sqrt(
        uncertainties[reference_index] ** 2
        + baseline["off_rms"] ** 2
    )
    z_counting = zeroed_reference / sigma_counting
    z_with_rms = zeroed_reference / sigma_with_rms

    summary_rows.append({
        "ke_range": ke_range,
        "zeroed_ref": zeroed_reference,
        "sigma_counting": sigma_counting,
        "off_rms": baseline["off_rms"],
        "z_counting": z_counting,
        "z_with_rms": z_with_rms,
    })
    print(
        f"{ke_range[0]:3.0f}-{ke_range[1]:3.0f} eV  "
        f"{zeroed_reference: .6g}  {sigma_counting: .3g}  "
        f"{baseline['off_rms']: .3g}  {z_counting: .3g}  "
        f"{z_with_rms: .3g}"
    )

print(f"Reference energy used: {reference_energy:.3f} eV")

## Best candidate window

The broad 190-220 eV kinetic-energy window gives the clearest local enhancement near 274 eV in the working analysis. The following plot shows only this baseline-subtracted trace. The peak estimates are descriptive values on the sampled photon-energy grid; they are not a resonance-model fit and do not account for a multiple-testing trials factor.

In [ ]:
values = candidate_traces["delta"][PEAK_KE_RANGE]
uncertainties = candidate_traces["sigma"][PEAK_KE_RANGE]
baseline = baseline_results[PEAK_KE_RANGE]

zeroed = values - baseline["off_mean"]
zeroed_error = np.sqrt(
    uncertainties ** 2 + baseline["off_mean_err"] ** 2
)
peak_mask = (
    np.isfinite(incident_energies)
    & np.isfinite(zeroed)
    & (incident_energies >= PEAK_ENERGY_WINDOW[0])
    & (incident_energies <= PEAK_ENERGY_WINDOW[1])
)
if np.count_nonzero(peak_mask) < 2:
    raise ValueError("Not enough points in PEAK_ENERGY_WINDOW")

peak_energies = incident_energies[peak_mask]
peak_values = zeroed[peak_mask]
peak_errors = zeroed_error[peak_mask]
maximum_index = int(np.nanargmax(peak_values))
peak_energy = peak_energies[maximum_index]
peak_value = peak_values[maximum_index]
peak_sigma = peak_errors[maximum_index]

positive_weights = np.maximum(peak_values, 0.0)
if np.nansum(positive_weights) > 0:
    centroid = (
        np.nansum(peak_energies * positive_weights)
        / np.nansum(positive_weights)
    )
    rms_width = np.sqrt(
        np.nansum(positive_weights * (peak_energies - centroid) ** 2)
        / np.nansum(positive_weights)
    )
else:
    centroid = np.nan
    rms_width = np.nan

above_half_maximum = peak_values >= 0.5 * peak_value
if np.count_nonzero(above_half_maximum) >= 2:
    fwhm_approx = (
        np.nanmax(peak_energies[above_half_maximum])
        - np.nanmin(peak_energies[above_half_maximum])
    )
else:
    fwhm_approx = np.nan

print(f"Peak maximum      : {peak_energy:.3f} eV")
print(f"Peak value        : {peak_value:.6g} +/- {peak_sigma:.3g} counts/uJ")
print(f"Positive centroid : {centroid:.3f} eV")
print(f"RMS width         : {rms_width:.3f} eV")
print(f"Approximate FWHM  : {fwhm_approx:.3f} eV")

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.errorbar(
    incident_energies,
    zeroed,
    yerr=zeroed_error,
    fmt="o-",
    lw=1.3,
    capsize=2,
    label=f"{PEAK_KE_RANGE[0]}-{PEAK_KE_RANGE[1]} eV",
)
ax.axhline(0.0, color="k", lw=0.8, alpha=0.6)
ax.axvline(
    peak_energy,
    color="tab:red",
    lw=1.0,
    ls="--",
    label=f"maximum {peak_energy:.2f} eV",
)
for lo, hi in BASELINE_ENERGY_RANGES:
    ax.axvspan(lo, hi, color="0.8", alpha=0.25)

ax.set_xlabel("incident photon energy (eV)")
ax.set_ylabel("Delta S minus weighted off-resonance mean")
ax.set_title("Candidate resonance-like trace: 190-220 eV KE")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Observations

- The low-GMD control shows that total GMD-normalised response is approximately linear, although the spectral shape is not perfectly invariant at every kinetic energy.
- The floating-family difference map and several neighbouring KE-window traces contain a localized enhancement near 274 eV incident photon energy. It is strongest when integrated over the broad 190-220 eV window, so it is treated as a candidate resonance-like feature rather than a single-bin fluctuation.
- The maps also contain broader GMD-dependent spectral distortions. The candidate therefore should not be interpreted as a confirmed resonance from this analysis alone; baseline choice, multiple tested windows, and run-to-run variation remain relevant.
- No unambiguous fixed-kinetic-energy Auger feature is claimed here. Moving photoelectron structure and stationary Auger structure would require further separation, and this ordinary yield analysis is not equivalent to the coincidence/covariance analysis of Schwickert *et al.*
- The integer-family signal decreases at the upper end of the scan. Reduced photon transmission or carbon contamination on the optics are possible experimental explanations, but they are not established by these plots.